# Download de XML de NF-e

Lê as chaves de acesso de um relatório do Excel. Consulta cada uma no site e salva os XMLs numa pasta.

**Como funciona:** abre uma janela real do Chrome/Chromium e repete o que você faria à mão: cola a chave, clica em *Buscar* e depois em *Baixar XML*.

- Se o site possuir uma verificação anti-robô (Cloudflare). Normalmente ela passa sozinha. Se aparecer um desafio na janela, **resolva você mesmo** e o script continua.
- Chaves já baixadas são puladas, então é só rodar de novo para completar o que falhou.
- No final, a célula 5 junta todos os XMLs numa planilha (cabeçalho + itens).

# 1. Bibliotecas

In [ ]:
import os
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager

# 2. Configurações - API Paga

In [ ]:
# --- CONFIGURAÇÕES ---
ARQUIVO_EXCEL = "notas_fiscais.xlsx"
COLUNA_DA_CHAVE = "CHAVE_ACESSO_NOTA"
PASTA_DOWNLOAD = "xmls_baixados"

# Intervalo de linhas a processar (numeração da planilha, a partir de 1 = primeira nota; FIM inclusivo).
# Rode em lotes pequenos para evitar bloqueio, ex.: 1-30, depois 31-60...
LINHA_INICIO = 760
LINHA_FIM = 770

PAUSA_ENTRE_NOTAS = (4, 9)      # segundos, sorteio aleatório entre notas
LOTE_PAUSA_LONGA = 10           # a cada N notas baixadas, faz uma pausa maior
PAUSA_LONGA = (30, 60)          # segundos
TEMPO_MAX_DOWNLOAD = 40         # segundos esperando o arquivo aparecer

import random, glob

download_dir = os.path.abspath(PASTA_DOWNLOAD)
os.makedirs(download_dir, exist_ok=True)

print(f"Lendo a planilha: {ARQUIVO_EXCEL}...")
df = pd.read_excel(ARQUIVO_EXCEL, dtype={COLUNA_DA_CHAVE: str})
if COLUNA_DA_CHAVE not in df.columns:
    raise ValueError(f"Coluna '{COLUNA_DA_CHAVE}' não encontrada. Disponíveis: {list(df.columns)}")

# Seleciona o intervalo (LINHA_FIM = None processa até o final)
df_lote = df.iloc[LINHA_INICIO - 1 : LINHA_FIM]
print(f"Total na planilha: {len(df)} | Processando linhas {LINHA_INICIO} a {LINHA_INICIO + len(df_lote) - 1}")

options = webdriver.ChromeOptions()
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
})
# options.add_argument("--headless=new")   # deixe comentado: o site costuma bloquear modo invisível

def arquivos_xml():
    return set(glob.glob(os.path.join(download_dir, "*.xml")))

def aguardar_download(antes):
    """Espera aparecer um .xml novo (sem .crdownload) e devolve o caminho."""
    fim = time.time() + TEMPO_MAX_DOWNLOAD
    while time.time() < fim:
        novos = arquivos_xml() - antes
        if novos and not glob.glob(os.path.join(download_dir, "*.crdownload")):
            return novos.pop()
        time.sleep(0.5)
    return None

from selenium.common.exceptions import ElementClickInterceptedException, ElementNotInteractableException

def fechar_popup(driver):
    """Fecha o aviso 'TEM NOVIDADE' (botão 'depois') que cobre a página."""
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, "button.swal-button--cancel")
        for b in btns:
            if b.is_displayed():
                driver.execute_script("arguments[0].click();", b)
                time.sleep(0.8)
    except Exception:
        pass

def clicar(driver, el):
    """Clique normal; se algo (anúncio/banner) cobrir o botão, clica via JavaScript."""
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(random.uniform(0.8, 1.5))
    try:
        el.click()
    except (ElementClickInterceptedException, ElementNotInteractableException):
        driver.execute_script("arguments[0].click();", el)

XP_BUSCAR = "//a[@id='searchBtn']"   # no site é um <a>, não <button>
XP_BAIXAR_XML = "//a[@id='downloadXmlBtn']"

print("Iniciando o navegador...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
baixadas = 0

try:
    for index, row in df_lote.iterrows():
        chave = "".join(ch for ch in str(row[COLUNA_DA_CHAVE]) if ch.isdigit())
        if len(chave) != 44:
            print(f"[{index+1}] Pulando: chave inválida ({chave}).")
            continue

        destino = os.path.join(download_dir, f"{chave}.xml")
        if os.path.exists(destino):
            print(f"[{index+1}] Já baixada, pulando: {chave}")
            continue

        print(f"[{index+1}] Processando chave: {chave}")
        try:
            driver.get("https://meudanfe.com.br/")
            wait = WebDriverWait(driver, 20)

            fechar_popup(driver)
            input_chave = wait.until(EC.element_to_be_clickable((By.ID, "searchTxt")))
            input_chave.clear()
            input_chave.send_keys(chave)

            clicar(driver, wait.until(EC.element_to_be_clickable((By.XPATH, XP_BUSCAR))))

            # NOVO: espera o botão "Baixar XML" aparecer e clica
            botao_xml = wait.until(EC.visibility_of_element_located((By.XPATH, XP_BAIXAR_XML)))
            antes = arquivos_xml()
            clicar(driver, botao_xml)

            novo = aguardar_download(antes)
            if novo is None:
                raise TimeoutError("o arquivo XML não apareceu na pasta")
            os.replace(novo, destino)      # renomeia para <chave>.xml
            baixadas += 1
            print(f"-> OK: {os.path.basename(destino)}")

        except Exception as e:
            print(f"❌ Erro na chave {chave}: {type(e).__name__}: {e}")
            continue

        # Pausas aleatórias para evitar bloqueio
        if baixadas and baixadas % LOTE_PAUSA_LONGA == 0:
            p = random.uniform(*PAUSA_LONGA)
            print(f"   (pausa longa de {p:.0f}s)")
            time.sleep(p)
        else:
            time.sleep(random.uniform(*PAUSA_ENTRE_NOTAS))

finally:
    print(f"\nProcesso finalizado! {baixadas} XML(s) baixado(s). Fechando o navegador...")
    driver.quit()
    print(f"Arquivos salvos na pasta: {download_dir}")

# 3 Baixar XML do Site: https://consultadanfe.com/ - está funcionando

In [ ]:
# --- CONFIGURAÇÕES ---
ARQUIVO_EXCEL = "notas_fiscais.xlsx"
COLUNA_DA_CHAVE = "CHAVE_ACESSO_NOTA"
PASTA_DOWNLOAD = "xmls_baixados"

# Intervalo de linhas a processar (numeração da planilha, a partir de 1 = primeira nota; FIM inclusivo).
# Rode em lotes pequenos para evitar bloqueio, ex.: 1-30, depois 31-60...
LINHA_INICIO = 1503
LINHA_FIM = 1552

PAUSA_ENTRE_NOTAS = (5, 9)      # segundos, sorteio aleatório entre notas
LOTE_PAUSA_LONGA = 10           # a cada N notas baixadas, faz uma pausa maior
PAUSA_LONGA = (30, 60)          # segundos
TEMPO_MAX_DOWNLOAD = 40         # segundos esperando o arquivo aparecer

import random, glob

download_dir = os.path.abspath(PASTA_DOWNLOAD)
os.makedirs(download_dir, exist_ok=True)

print(f"Lendo a planilha: {ARQUIVO_EXCEL}...")
df = pd.read_excel(ARQUIVO_EXCEL, dtype={COLUNA_DA_CHAVE: str})
if COLUNA_DA_CHAVE not in df.columns:
    raise ValueError(f"Coluna '{COLUNA_DA_CHAVE}' não encontrada. Disponíveis: {list(df.columns)}")

# Seleciona o intervalo (LINHA_FIM = None processa até o final)
df_lote = df.iloc[LINHA_INICIO - 1 : LINHA_FIM]
print(f"Total na planilha: {len(df)} | Processando linhas {LINHA_INICIO} a {LINHA_INICIO + len(df_lote) - 1}")

options = webdriver.ChromeOptions()
options.add_experimental_option("prefs", {
    "download.default_directory": download_dir,
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True,
})
# options.add_argument("--headless=new")   # deixe comentado: o site costuma bloquear modo invisível

def arquivos_xml():
    return set(glob.glob(os.path.join(download_dir, "*.xml")))

def aguardar_download(antes):
    """Espera aparecer um .xml novo (sem .crdownload) e devolve o caminho."""
    fim = time.time() + TEMPO_MAX_DOWNLOAD
    while time.time() < fim:
        novos = arquivos_xml() - antes
        if novos and not glob.glob(os.path.join(download_dir, "*.crdownload")):
            return novos.pop()
        time.sleep(0.5)
    return None

from selenium.common.exceptions import ElementClickInterceptedException, ElementNotInteractableException

def fechar_popup(driver):
    """Fecha o aviso 'TEM NOVIDADE' (botão 'depois') que cobre a página."""
    try:
        btns = driver.find_elements(By.CSS_SELECTOR, "button.swal-button--cancel")
        for b in btns:
            if b.is_displayed():
                driver.execute_script("arguments[0].click();", b)
                time.sleep(0.8)
    except Exception:
        pass

def clicar(driver, el):
    """Clique normal; se algo (anúncio/banner) cobrir o botão, clica via JavaScript."""
    driver.execute_script("arguments[0].scrollIntoView({block:'center'});", el)
    time.sleep(random.uniform(0.8, 1.5))
    try:
        el.click()
    except (ElementClickInterceptedException, ElementNotInteractableException):
        driver.execute_script("arguments[0].click();", el)

XP_BUSCAR = "//button[normalize-space(.)='Imprimir DANFE']"
XP_BAIXAR_XML = "//button[normalize-space(.)='Baixar XML']"   # botão dentro da janela 'DANFE Gerado com Sucesso'

print("Iniciando o navegador...")
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
baixadas = 0

try:
    for index, row in df_lote.iterrows():
        chave = "".join(ch for ch in str(row[COLUNA_DA_CHAVE]) if ch.isdigit())
        if len(chave) != 44:
            print(f"[{index+1}] Pulando: chave inválida ({chave}).")
            continue

        destino = os.path.join(download_dir, f"{chave}.xml")
        if os.path.exists(destino):
            print(f"[{index+1}] Já baixada, pulando: {chave}")
            continue

        print(f"[{index+1}] Processando chave: {chave}")
        try:
            driver.get("https://consultadanfe.com/")
            wait = WebDriverWait(driver, 30)

            fechar_popup(driver)
            input_chave = wait.until(EC.element_to_be_clickable((By.ID, "accessKey")))
            input_chave.clear()
            input_chave.send_keys(chave)

            clicar(driver, wait.until(EC.element_to_be_clickable((By.XPATH, XP_BUSCAR))))

            # NOVO: espera o botão "Baixar XML" aparecer e clica
            botao_xml = wait.until(EC.visibility_of_element_located((By.XPATH, XP_BAIXAR_XML)))
            antes = arquivos_xml()
            clicar(driver, botao_xml)

            novo = aguardar_download(antes)
            if novo is None:
                raise TimeoutError("o arquivo XML não apareceu na pasta")
            os.replace(novo, destino)      # renomeia para <chave>.xml
            baixadas += 1
            print(f"-> OK: {os.path.basename(destino)}")

        except Exception as e:
            print(f"❌ Erro na chave {chave}: {type(e).__name__}: {e}")
            continue

        # Pausas aleatórias para evitar bloqueio
        if baixadas and baixadas % LOTE_PAUSA_LONGA == 0:
            p = random.uniform(*PAUSA_LONGA)
            print(f"   (pausa longa de {p:.0f}s)")
            time.sleep(p)
        else:
            time.sleep(random.uniform(*PAUSA_ENTRE_NOTAS))

finally:
    print(f"\nProcesso finalizado! {baixadas} XML(s) baixado(s). Fechando o navegador...")
    driver.quit()
    print(f"Arquivos salvos na pasta: {download_dir}")

# 4 Importar para o sistema SISGI

In [ ]:
# Envia os XMLs da pasta "xmls_baixados" pelo botão "Importar XML" de /despesas_lista.
# Abre o Chrome, você faz o login na janela e o script continua sozinho.
# Agrupa os arquivos pelo mês de emissão (o SISGI só aceita notas do mês de competência) e envia em lotes de 20.
# Use SIMULAR = True para ver o plano sem enviar nada.

# --- CONFIGURAÇÕES ---
URL_SISGI = "http://sisgi.hmg.seduc.mt.gov.br/despesas_lista"
PASTA_XML = "xmls_baixados"          # mesma pasta da célula 3
ARQUIVOS_POR_LOTE = 20               # limite do sistema: 20 arquivos por vez
TEMPO_MAX_LOGIN = 300                # segundos esperando você fazer login na janela
TEMPO_MAX_IMPORTAR = 180             # segundos esperando o sistema processar cada lote
MES_FILTRO = None                    # ex.: "2025-03" para importar só esse mês; None = todos
LIMITE_ARQUIVOS = None               # ex.: 40 para testar com poucos arquivos; None = todos
SIMULAR = False                      # True = só mostra o plano, sem importar nada
ARQUIVO_LOG = "sisgi_importados.txt" # guarda o que já foi enviado (evita reenviar)

import os, glob, time
import xml.etree.ElementTree as ET
from collections import defaultdict
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

pasta = os.path.abspath(PASTA_XML)

# 1) Lê a data de emissão de cada XML -> o sistema só aceita notas do mês de competência informado
def mes_emissao(caminho):
    for _, el in ET.iterparse(caminho, events=("end",)):
        tag = el.tag.split("}")[-1]
        if tag in ("dhEmi", "dEmi") and el.text:
            return el.text.strip()[:7]          # AAAA-MM
    return None

ja_enviados = set()
if os.path.exists(ARQUIVO_LOG):
    with open(ARQUIVO_LOG, encoding="utf-8") as f:
        ja_enviados = {l.strip() for l in f if l.strip()}

por_mes, sem_data = defaultdict(list), []
for arq in sorted(glob.glob(os.path.join(pasta, "*.xml"))):
    if os.path.basename(arq) in ja_enviados:
        continue
    try:
        mes = mes_emissao(arq)
    except ET.ParseError:
        mes = None
    if not mes:
        sem_data.append(arq); continue
    if MES_FILTRO and mes != MES_FILTRO:
        continue
    por_mes[mes].append(arq)

if LIMITE_ARQUIVOS:
    restante = LIMITE_ARQUIVOS
    for mes in sorted(por_mes):
        por_mes[mes] = por_mes[mes][:max(restante, 0)]
        restante -= len(por_mes[mes])

total = sum(len(v) for v in por_mes.values())
print(f"Pasta: {pasta}")
print(f"XMLs a importar: {total} | já enviados antes: {len(ja_enviados)} | ilegíveis/sem data: {len(sem_data)}")
for mes in sorted(por_mes):
    print(f"  {mes}: {len(por_mes[mes])} arquivo(s)")
for a in sem_data:
    print(f"  ⚠ sem data de emissão: {os.path.basename(a)}")

if total == 0 or SIMULAR:
    print("\nNada a enviar." if total == 0 else "\nSIMULAR=True: nenhum arquivo foi enviado.")
else:
    # 2) Abre o Chrome e espera o login (faça o login você mesmo na janela que abrir)
    opcoes = webdriver.ChromeOptions()
    opcoes.set_capability("unhandledPromptBehavior", "ignore")   # não aborta se aparecer alert() no site
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opcoes)
    driver.maximize_window()
    ok_total = 0
    try:
        driver.get(URL_SISGI)
        print("\n>>> Faça o login na janela do Chrome. O script continua sozinho quando a lista de despesas aparecer.")
        alertas_vistos = set()
        def login_ok(d):
            try:
                return bool(d.find_elements(By.ID, "btnAbrirImportarXml"))
            except Exception as e:   # ex.: alert aberto na tela
                try:
                    txt = d.switch_to.alert.text
                    if txt not in alertas_vistos:
                        alertas_vistos.add(txt)
                        print(f"   (alerta do site: {txt!r} - clique OK na janela)")
                except Exception:
                    pass
                return False
        WebDriverWait(driver, TEMPO_MAX_LOGIN, poll_frequency=1).until(login_ok)
        print("Login detectado. Iniciando importação...\n")

        def abrir_modal():
            wait = WebDriverWait(driver, 20)
            btn = wait.until(EC.element_to_be_clickable((By.ID, "btnAbrirImportarXml")))
            driver.execute_script("arguments[0].click();", btn)
            wait.until(EC.visibility_of_element_located((By.ID, "modal_arquivos_xml")))

        def mensagens_da_pagina():
            """Captura avisos/alertas visíveis (sucesso, erro, notas ignoradas)."""
            seletores = ".alert, .toast, .swal2-popup, .swal-modal, .mensagem, .flash, [role=alert]"
            textos = []
            for e in driver.find_elements(By.CSS_SELECTOR, seletores):
                try:
                    if e.is_displayed() and e.text.strip():
                        textos.append(e.text.strip())
                except Exception:
                    pass
            return textos

        for mes in sorted(por_mes):
            arquivos = por_mes[mes]
            for i in range(0, len(arquivos), ARQUIVOS_POR_LOTE):
                lote = arquivos[i:i + ARQUIVOS_POR_LOTE]
                print(f"[{mes}] lote {i // ARQUIVOS_POR_LOTE + 1}: {len(lote)} arquivo(s)")
                driver.get(URL_SISGI)
                abrir_modal()

                # competência (input type=month -> valor AAAA-MM) com eventos para o site perceber
                comp = driver.find_element(By.ID, "modal_competencia")
                driver.execute_script(
                    "arguments[0].value = arguments[1];"
                    "arguments[0].dispatchEvent(new Event('input',{bubbles:true}));"
                    "arguments[0].dispatchEvent(new Event('change',{bubbles:true}));", comp, mes)

                # anexa os arquivos (caminhos separados por quebra de linha)
                driver.find_element(By.ID, "modal_arquivos_xml").send_keys("\n".join(lote))

                antigo = driver.find_element(By.TAG_NAME, "html")
                driver.execute_script("arguments[0].click();",
                                      driver.find_element(By.ID, "btnConfirmarImportarXml"))

                # o SISGI responde com alert() e/ou recarrega a página: lê, confirma e segue
                fim = time.time() + TEMPO_MAX_IMPORTAR
                ultimo_alerta = None
                while time.time() < fim:
                    try:
                        al = driver.switch_to.alert
                        txt = al.text
                        al.accept()
                        print("   -> SISGI:", txt.replace("\n", " | "))
                        ultimo_alerta = time.time()
                        time.sleep(1.5)
                        continue
                    except Exception:
                        pass
                    try:
                        antigo.is_enabled()          # lança erro se a página já recarregou
                    except Exception as ex:
                        if "stale" in type(ex).__name__.lower():
                            time.sleep(1.5)
                            break
                    if ultimo_alerta and time.time() - ultimo_alerta > 5:
                        break                        # alerta tratado e nada mais aconteceu
                    time.sleep(0.5)
                else:
                    print("   ⚠ sem resposta no tempo esperado; verifique na tela.")

                try:
                    for m in mensagens_da_pagina():
                        print("   ->", m.replace("\n", " | "))
                except Exception:
                    pass

                with open(ARQUIVO_LOG, "a", encoding="utf-8") as f:
                    for a in lote:
                        f.write(os.path.basename(a) + "\n")
                ok_total += len(lote)
                time.sleep(2)
    finally:
        print(f"\nFinalizado. {ok_total} arquivo(s) enviado(s) ao SISGI.")
        # avisa na tela do navegador e só fecha quando você clicar em OK
        try:
            driver.execute_script(
                "setTimeout(function(){alert(arguments[0]);}, 100);",
                f"Importação finalizada!\n\n{ok_total} arquivo(s) enviado(s) ao SISGI.\n\nClique em OK para fechar o navegador.")
            fim = time.time() + 600            # espera até 10 min pelo seu OK
            apareceu = False
            while time.time() < fim:
                try:
                    driver.switch_to.alert.text
                    apareceu = True
                except Exception:
                    if apareceu:
                        break                  # alerta sumiu = você clicou em OK
                time.sleep(0.5)
        except Exception as e:
            print("Não foi possível exibir o aviso no navegador:", e)
        print("Fechando o navegador...")
        driver.quit()